In [ ]:
import numpy as np
import os
import time
from collections import defaultdict
import torch
import torch.nn as nn
import torch.utils.data as  data
from torch.utils.data import DataLoader, Dataset



In [ ]:
def sequence_data(num_sequence = 2**6):

  np.random.seed(42)

  sample = []



  for _ in range(num_sequence):

      num_token = np.random.randint(1, 12)
      correct_sequence = ["a"] * num_token + ["b"] * num_token + ["EOS"]

      sample.append(correct_sequence)
  return sample


sequence = sequence_data()

print("the sequence of this dataset:")
print(sequence[0])



the sequence of this dataset:
['a', 'a', 'a', 'a', 'a', 'a', 'a', 'b', 'b', 'b', 'b', 'b', 'b', 'b', 'EOS']


In [ ]:
def sequence_to_dict(sequences):

  flatten = lambda l :[item for sublist in l for item in sublist]

  all_word = flatten(sequence)


  word_count = defaultdict(int)
  for word in flatten(sequence):
    word_count[word] += 1

  word_count = sorted(list(word_count.items()), key = lambda l: -l[1])

  unique_word = [item[0]for item in word_count]



  unique_word.append("UNK")


  num_sentences, vocal_size = len(sequence), len(unique_word)

  word_to_idx = defaultdict(lambda: vocal_size-1)
  idx_to_word = defaultdict(lambda:"UNK")


  for idx, word in enumerate(unique_word):
    word_to_idx[word] = idx
    idx_to_word[idx] = word



  return word_to_idx, idx_to_word, num_sentences, vocal_size


word_to_idx, idx_to_word, num_sequence, vocal_size = sequence_to_dict(sequence)

print(f"we have {num_sequence} sentences")
print(f" we have {vocal_size}, vocal_size")
print(word_to_idx)
print(idx_to_word)

we have 64 sentences
 we have 4, vocal_size
defaultdict(<function sequence_to_dict.<locals>.<lambda> at 0x7d258ab9d080>, {'a': 0, 'b': 1, 'EOS': 2, 'UNK': 3})
defaultdict(<function sequence_to_dict.<locals>.<lambda> at 0x7d258ab9d120>, {0: 'a', 1: 'b', 2: 'EOS', 3: 'UNK'})


In [ ]:
### creating the dataset
class Dataset(data.Dataset):
  def __init__(self, input, target):
    super().__init__()
    self.input = input
    self.target = target

  def __len__(self):
    return len(self.target)


  def __getitem__(self, index):
    X = self.input[index]
    Y = self.target[index]




    return X, Y



def create_datasets(sequence, dataset_class, p_train = 0.6, p_val = 0.2, p_test = 0.2):
  num_train = int(len(sequence)*p_train)
  num_val = int(len(sequence)*p_val)
  num_test = int(len(sequence)*p_test)



  sequence_train = sequence[:num_train]
  sequence_val = sequence[num_train:num_train + num_val]
  sequence_test = sequence[-num_test:]

  return sequence_train, sequence_val, sequence_test


def get_input_values(sequence):
  input, target = [], []


  for seq in sequence:
    input.append(seq[:-1])
    target.append(seq[1:])

  return input, target


# Call create_datasets to initialize the sequence splits
sequence_train, sequence_val, sequence_test = create_datasets(sequence, Dataset)

input_train, target_train = get_input_values(sequence_train)
input_val, target_val = get_input_values(sequence_val)
input_test, target_test = get_input_values(sequence_test)


training_set = Dataset(input_train, target_train)
validation_set = Dataset(input_val, target_val)
testing_set = Dataset(input_test, target_test)


print(f"the length of the training set is {len(training_set)}")
print(f"the length of the validation set is {len(validation_set)}")
print(f"the length of the testing set is {len(testing_set)}")

the length of the training set is 38
the length of the validation set is 12
the length of the testing set is 12


In [ ]:
def one_hot_encode(idx, vocal_size):

  one_hot = torch.zeros(vocal_size)

  one_hot[idx] = 1.0

  return one_hot

def one_hot_sequence(sequence, vocal_size):

  # Create a list of one-hot encoded tensors and stack them
  encoded_list = [one_hot_encode(word_to_idx[word], vocal_size) for word in sequence]
  encoding = torch.stack(encoded_list) # Now encoding will be a torch.Tensor of shape (len(sequence), vocal_size)

  # Reshape to (sequence_length, vocal_size, 1) to match output from forward pass
  encoding = encoding.reshape(encoding.shape[0], encoding.shape[1], 1)

  return encoding


test_encod = one_hot_encode(word_to_idx["a"],vocal_size)
print(f"the shape of this {test_encod.shape}")

test_sentences = one_hot_sequence(["a", "b"], vocal_size)

print(f" the shape of the test_sentences is {test_sentences.shape}")
print(f" the test_sentences is \n{test_sentences}")

the shape of this torch.Size([4])
 the shape of the test_sentences is torch.Size([2, 4, 1])
 the test_sentences is 
tensor([[[1.],
         [0.],
         [0.],
         [0.]],

        [[0.],
         [1.],
         [0.],
         [0.]]])


In [ ]:
### implementing RNN

hidden_size = 500
vocal_size = len(word_to_idx)


## init params for the rnn ####
def init_orthgonal(params):
  if type(params) == nn.Linear:
    nn.init.orthogonal_(params.weight)
  elif type(params) == nn.Linear:
    nn.init.orthogonal_(params.bias)


def init_layer(hidden_size, vocal_size):

  U = np.random.randn(hidden_size, vocal_size) * 0.01 # Initialize with small random values
  W = np.random.randn(hidden_size, hidden_size) * 0.01 # Initialize with small random values
  V = np.random.randn(vocal_size, hidden_size) * 0.01 # Initialize with small random values
  B_hidden = np.zeros((hidden_size, 1))
  B_output = np.zeros((vocal_size, 1))



  # The init_orthgonal function is designed for nn.Linear modules
  # and does not return a value for numpy arrays, causing them to be None.
  # Removing these calls to correctly return the numpy arrays.
  # u = init_orthgonal(U)
  # w = init_orthgonal(W)
  # v = init_orthgonal(V)


  return U, W, V, B_hidden, B_output



params = init_layer(hidden_size, vocal_size)

print(f"the shape of U is {params[0].shape}")
print(f"the shape of W is {params[1].shape}")
print(f"the shape of V is {params[2].shape}")
print(f" the shape of B_hidden is {params[3].shape}")
print(f"the shape of B_output is {params[4].shape}")

the shape of U is (500, 4)
the shape of W is (500, 500)
the shape of V is (4, 500)
 the shape of B_hidden is (500, 1)
the shape of B_output is (4, 1)


In [ ]:
### implementing the forward pass

def softmax(x):
    # Apply softmax to a 2D array (logits, 1), ensuring numerical stability
    # x is expected to be (vocal_size, 1)
    e_x = np.exp(x - np.max(x, axis=0, keepdims=True))
    return e_x / np.sum(e_x, axis=0, keepdims=True)

def forward(self, input_sequence, params):

  U, W, V, B_hidden, B_output = params

  hidden_state = np.zeros((hidden_size, 1))
  output_sequence = []
  hidden_states_list = [hidden_state] # Store initial hidden state (h0)

  for t in range(input_sequence.shape[0]):
    # input_sequence[t] will be of shape (1, vocal_size, 1) from one_hot_sequence
    current_input_t = input_sequence[t].reshape(vocal_size, 1) # Reshape to (vocal_size, 1)

    # Correct RNN equation: h_t = tanh(U @ x_t + W @ h_{t-1} + B_hidden)
    hidden_state = np.tanh(U @ current_input_t + W @ hidden_state + B_hidden)
    output_logits = V @ hidden_state + B_output # These are the raw logits
    output_probs = softmax(output_logits) # Apply softmax to get probabilities
    output_sequence.append(output_probs)
    hidden_states_list.append(hidden_state) # Store hidden state after each step (h1, h2, ...)

  return np.array(output_sequence), hidden_states_list # Return both outputs (probabilities) and all hidden states

In [ ]:
### implementing the backward pass

def backward(self, input_sequence, target_sequence, output_sequence_pred, hidden_states_list, params):
  U, W, V, B_hidden, B_output = params

  dU, dW, dV, dB_hidden, dB_output = np.zeros_like(U), np.zeros_like(W), np.zeros_like(V), np.zeros_like(B_hidden), np.zeros_like(B_output)

  # Initialize d_hidden_state_error, which is the error propagated from the *next* hidden state
  # We start with no error from the 'future' at the last time step
  d_hidden_state_error_from_next_h = np.zeros_like(hidden_states_list[0]) # Shape (hidden_size, 1)

  # Iterate backward through time steps
  # hidden_states_list contains h0, h1, ..., hT. We need h_t for t-1 and t
  # input_sequence contains x0, x1, ..., x_{T-1}
  # output_sequence_pred contains y_pred_0, y_pred_1, ..., y_pred_{T-1} (now probabilities)

  # Loop from T-1 down to 0
  for t in reversed(range(len(input_sequence))):
    # Get the actual hidden state at time t and t-1 from the forward pass
    hidden_state_t = hidden_states_list[t+1] # h_t (after processing x_t)
    hidden_state_prev = hidden_states_list[t] # h_{t-1} (before processing x_t)

    # Get the current input x_t
    current_input_t = input_sequence[t].reshape(vocal_size, 1) # Reshape to (vocal_size, 1)

    # Get target and predicted output (probabilities) for current time step
    target_t = target_sequence[t] # Shape: (vocal_size, 1)
    output_pred_t = output_sequence_pred[t] # Shape: (vocal_size, 1) (probabilities from softmax)

    # Calculate output error (dL/d_logits) for Cross-Entropy with Softmax
    # This is simply (output_probabilities - target_one_hot)
    output_error_t = output_pred_t - target_t # Shape: (vocal_size, 1)

    # Gradients for V and B_output
    # dV = (dL/d_logits) @ hidden_state_t.T
    dV += output_error_t @ hidden_state_t.T # (vocal_size, 1) @ (1, hidden_size) = (vocal_size, hidden_size)
    # dB_output = dL/d_logits
    dB_output += output_error_t # (vocal_size, 1)

    # Error propagated back from output to the hidden state at time t
    # dL/d_hidden_state_t = V.T @ (dL/d_logits)
    d_hidden_state_error_from_output = V.T @ output_error_t # (hidden_size, vocal_size) @ (vocal_size, 1) = (hidden_size, 1)

    # Total error for hidden state at time t: error from output + error from next hidden state
    d_hidden_state_error_at_t = d_hidden_state_error_from_output + d_hidden_state_error_from_next_h

    # Derivative of tanh: (1 - h_t^2)
    tanh_prime = (1 - hidden_state_t**2) # Element-wise: (hidden_size, 1)

    # Error before tanh activation at current time step t
    d_pre_tanh = d_hidden_state_error_at_t * tanh_prime # Element-wise: (hidden_size, 1)

    # Gradients for U, W, B_hidden
    # dU depends on x_t
    dU += d_pre_tanh @ current_input_t.T # (hidden_size, 1) @ (1, vocal_size) = (hidden_size, vocal_size)
    # dW depends on h_{t-1}
    dW += d_pre_tanh @ hidden_state_prev.T # (hidden_size, 1) @ (1, hidden_size) = (hidden_size, hidden_size)
    # dB_hidden
    dB_hidden += d_pre_tanh # (hidden_size, 1)

    # Propagate error backward to the previous hidden state (for next iteration)
    # Error through W to h_{t-1} + error through U (which we don't need to propagate back as input is fixed)
    d_hidden_state_error_from_next_h = W.T @ d_pre_tanh # (hidden_size, hidden_size) @ (hidden_size, 1) = (hidden_size, 1)

  return dU, dW, dV, dB_hidden, dB_output

# --- Dummy data for testing the backward function (removed to avoid clutter during training loop execution) ---
# The following lines are commented out or removed as they were for testing and cause errors during normal execution.
# Removed the dummy data and backward call that was causing the IndentationError in the previous execution.
# This ensures the cell only defines the backward function.

In [ ]:
### optimizer ####

def optimizer(params, grads, lr = 1e-4, clip_value=5.0):

  # Calculate total gradient norm
  total_norm = 0
  for grad in grads:
    total_norm += np.sum(grad**2)
  total_norm = np.sqrt(total_norm)

  # Apply clipping if norm exceeds threshold
  clip_factor = 1.0
  if total_norm > clip_value:
    clip_factor = clip_value / (total_norm + 1e-6) # Add epsilon for numerical stability

  # Update parameters
  new_params = [] # Create a new list to store updated parameters
  for param, grad in zip(params, grads):
    new_params.append(param - lr * grad * clip_factor) # Apply clipped gradient

  return tuple(new_params) # Return as tuple

In [ ]:
## Ensure all preceding cells, especially the one defining init_layer, hidden_size, and vocal_size (e.g., cell 9LYl9VwvPPdJ), are executed before running this cell.
num_epochs = 3000

# Initialize params using init_layer
params = init_layer(hidden_size = hidden_size, vocal_size = vocal_size)

training_losses_history, validation_losses_history = [], []


for i in range(num_epochs):

  training_epoch_loss = 0
  validation_epoch_loss = 0


  for input_data, target_data in validation_set: # Renamed loop variables to avoid conflict with 'input' argument
    input_hot_sentence = one_hot_sequence(input_data, vocal_size).numpy() # Convert to NumPy array
    target_hot_sentence = one_hot_sequence(target_data, vocal_size).numpy() # Convert to NumPy array

    # Call forward correctly, passing the one-hot encoded input_hot_sentence
    output_pred_sequence, hidden_states_list = forward(None, input_hot_sentence, params) # output_pred_sequence are now probabilities

    # Calculate Cross-Entropy Loss (NumPy implementation)
    # Add a small epsilon to np.log to prevent log(0) which results in -inf
    epsilon = 1e-8
    # Sum across vocal_size and the last dimension (which is 1 due to reshaping in one_hot_sequence)
    current_loss_val = -np.sum(target_hot_sentence * np.log(np.clip(output_pred_sequence, epsilon, 1.0)), axis=(1, 2))
    loss = np.mean(current_loss_val) # Average over sequence items

    validation_epoch_loss += loss # Directly add NumPy scalar

  validation_losses_history.append(validation_epoch_loss)


  for input_data, target_data in training_set: # Renamed loop variables
    input_hot_sentence = one_hot_sequence(input_data, vocal_size).numpy() # Convert to NumPy array
    target_hot_sentence = one_hot_sequence(target_data, vocal_size).numpy() # Convert to NumPy array

    # Call forward correctly
    output_pred_sequence, hidden_states_list = forward(None, input_hot_sentence, params) # output_pred_sequence are now probabilities

    # Calculate training loss (Cross-Entropy)
    epsilon = 1e-4
    current_loss_train = -np.sum(target_hot_sentence * np.log(np.clip(output_pred_sequence, epsilon, 1.0)), axis=(1, 2))
    training_loss_current = np.mean(current_loss_train)

    # Call the updated backward function with hidden_states_list
    grad_dU, grad_dW, grad_dV, grad_dB_hidden, grad_dB_output = backward(None, input_hot_sentence, target_hot_sentence, output_pred_sequence, hidden_states_list, params)
    grads = (grad_dU, grad_dW, grad_dV, grad_dB_hidden, grad_dB_output)

    params = optimizer(params, grads)

    training_epoch_loss += training_loss_current

  training_losses_history.append(training_epoch_loss)



print(f" the training_losses_history {training_losses_history}")
print(f" the validation_losses_history {validation_losses_history}")

 the training_losses_history [np.float64(52.57486438619892), np.float64(52.32184823237867), np.float64(52.07188014141163), np.float64(51.82490610737354), np.float64(51.580873021827585), np.float64(51.33972860249221), np.float64(51.10142132557819), np.float64(50.865900361549045), np.float64(50.63311551407784), np.float64(50.40301716199069), np.float64(50.175556204005375), np.float64(49.95068400608831), np.float64(49.72835235126975), np.float64(49.50854419404878), np.float64(49.2912971157221), np.float64(49.07661050975367), np.float64(48.86442976655893), np.float64(48.65470074664034), np.float64(48.447369735900466), np.float64(48.242383403207796), np.float64(48.03968876010789), np.float64(47.83923312258829), np.float64(47.640964074819074), np.float64(47.44482943480533), np.float64(47.250777221901075), np.float64(47.0587556261474), np.float64(46.868712979409445), np.float64(46.68059772830005), np.float64(46.49435840888836), np.float64(46.30994362320406), np.float64(46.1273020175577), np.f

In [ ]:
#### LSTM  #####

d_size = hidden_size + vocal_size


def init_orthogonal(params):
  if type(params) == nn.Linear:
    nn.init.orthogonal_(params.weight)
  elif type(params) == nn.Linear:
    nn.init.orthogonal_(params.bias)



def init_layer_lstm(hidden_size, vocal_size, d_size):

  U = np.random.randint(1, 10, (hidden_size, d_size))

  W = np.random.randint(1, 10, (hidden_size, hidden_size))

  V = np.random.randint(1, 10, (vocal_size, hidden_size))

  B_hidden = np.zeros((hidden_size, 1))

  B_output = np.zeros((vocal_size, 1))

  return U, W, V, B_hidden, B_output




params = init_layer_lstm(hidden_size = hidden_size, vocal_size = vocal_size, d_size = d_size)

print(f"the value of U is {params[0].shape}")
print(f"the value of V is {params[1].shape}")
print(f"the value of W is {params[2].shape}")
print(f"the value of B_hidden is {params[3].shape}")
print(f"the value of B_output is {params[4].shape}")




the value of U is (500, 504)
the value of V is (500, 500)
the value of W is (4, 500)
the value of B_hidden is (500, 1)
the value of B_output is (4, 1)


In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def forward_lstm(input_sequence_data, h_prev, C_prev, p):

  assert h_prev.shape == (hidden_size, 1)
  assert C_prev.shape == (hidden_size, 1)


  f_w, f_b, i_w, i_b, o_w, o_b, u_w, u_b, V_out, B_output_out = p

  x_s, f_s_list, i_s_list, g_s_list = [], [], [], []
  C_s_list, o_s_list, h_s_list = [], [], []
  v_s, output_s_list =  [], []


  h_s_list.append(h_prev) # Store initial hidden state h0
  C_s_list.append(C_prev) # Store initial cell state C0


  for t in range(input_sequence_data.shape[0]): # Iterate over the length of the input sequence
    x = input_sequence_data[t] # Access current input vector
    x_s.append(x)

    # Concatenate h_prev and x for gate inputs
    concat_hx = np.vstack((h_prev, x))

    # Calculate gate activations using sigmoid
    f = sigmoid(np.dot(f_w, concat_hx) + f_b) # Forget gate
    f_s_list.append(f)

    i = sigmoid(np.dot(i_w, concat_hx) + i_b) # Input gate
    i_s_list.append(i)

    o = sigmoid(np.dot(o_w, concat_hx) + o_b) # Output gate
    o_s_list.append(o)

    g = np.tanh(np.dot(u_w, concat_hx) + u_b) # Candidate cell state
    g_s_list.append(g)

    C_prev = f * C_prev + i * g # Update cell state
    C_s_list.append(C_prev)

    h_prev = o * np.tanh(C_prev) # Update hidden state
    h_s_list.append(h_prev)

    # Output layer processing (V @ h_prev + B_output_out, then softmax)
    output_logits = V_out @ h_prev + B_output_out
    soft = softmax(output_logits) # Use the globally defined softmax function
    v_s.append(soft)
    output_s_list.append(soft)


  return x_s, f_s_list, i_s_list, o_s_list, g_s_list, h_s_list, v_s, output_s_list, C_s_list


input_data_sample, target = testing_set[1]

encoded_input_sequence = one_hot_sequence(input_data_sample, vocal_size)
encoded_input_sequence = encoded_input_sequence.numpy()


h_prev = np.zeros((hidden_size, 1))
c_prev = np.zeros((hidden_size, 1))

# The original 'params' from cell 0V-3kT2kuqOa only contains 5 elements.
# The forward_lstm function expects 8 LSTM gate parameters + 2 for the output layer.
# Explicitly initializing these parameters here to resolve the ValueError and for full LSTM functionality.

f_w = np.random.randn(hidden_size, d_size) * 0.01
f_b = np.zeros((hidden_size, 1))
i_w = np.random.randn(hidden_size, d_size) * 0.01
i_b = np.zeros((hidden_size, 1))
o_w = np.random.randn(hidden_size, d_size) * 0.01
o_b = np.zeros((hidden_size, 1))
u_w = np.random.randn(hidden_size, d_size) * 0.01
u_b = np.zeros((hidden_size, 1))
V_out = np.random.randn(vocal_size, hidden_size) * 0.01 # Output weights
B_output_out = np.zeros((vocal_size, 1)) # Output bias

lstm_params = (f_w, f_b, i_w, i_b, o_w, o_b, u_w, u_b, V_out, B_output_out)


z_s, f_s, i_s, o_s, g_s, h_s, v_s, outputs, C_s = forward_lstm(encoded_input_sequence, h_prev, c_prev, lstm_params)


print(f"the shape of z_s (x_s) is {np.array(z_s).shape}")
print(f"the shape of f_s is {np.array(f_s).shape}")
print(f"the shape of i_s is {np.array(i_s).shape}")
print(f"the shape of o_s is {np.array(o_s).shape}")
print(f"the shape of g_s is {np.array(g_s).shape}")
print(f"the shape of h_s is {np.array(h_s).shape}")
print(f"the shape of v_s is {np.array(v_s).shape}")
print(f"the shape of outputs is {np.array(outputs).shape}")
print(f"the shape of C_s is {np.array(C_s).shape}")

the shape of z_s (x_s) is (14, 4, 1)
the shape of f_s is (14, 500, 1)
the shape of i_s is (14, 500, 1)
the shape of o_s is (14, 500, 1)
the shape of g_s is (14, 500, 1)
the shape of h_s is (15, 500, 1)
the shape of v_s is (14, 4, 1)
the shape of outputs is (14, 4, 1)
the shape of C_s is (15, 500, 1)


In [ ]:
def backward(z, f, i, g, C, o, h, v, outputs, targets, p = params):

  w_f, w_i, w_g, w_o, w_v, b_f, b_i, b_g, b_o, b_v = params

  ## initializing the graident of the values

  d_w_f = np.zeros_like(w_f)
  d_b_f = np.zeros_like(b_f)

  d_w_i = np.zeros_like(w_i)
  d_b_i = np.zeros_like(b_i)

  d_w_g = np.zeros_like(w_g)
  d_b_g = np.zeros_like(b_g)

  d_w_o = np.zeros_like(w_o)
  d_b_o = np.zeos_like(b_o)

  d_w_v = np.zeros_like(w_v)
  d_b_v = np.zeros_like(b_v)



  dcell_next = np.zeros_like(h[0])
  dhidden_next = np.zeros_like(h[0])


  loss = 0



  for t in reversed(range(len(outputs))):

    loss += -np.mean(np.log(outputs[t] * target[t])) ### computing the cross_entropy_loss

    c_prev = C[t-1]


    dv = np.copy(outputs[t])
    dv[np.argmax(targets[t])] -= 1


  d_w_g += np.dot(dv, h[t].T)
  d_b_g += dv



  d_w_v += np.dot(w_v.T, dv)
  d_w_v += dhidden_next
  do = d_w_v * np.tanh(C[t])




